# Session 6 · Text I: Finding "Good"

**Goal:** a supervisor asks for every offender whose conduct notes describe **good behavior**. We'll try seven increasingly clever searches and score each one against an answer key.

**How to use this notebook**
- Click a gray code cell and press **Shift + Enter** to run it. Go top to bottom.
- 🧪 **Your turn** cells contain a `___` blank. Replace it with your answer and run the cell; it tells you whether you're right. Answers are at the bottom if you're stuck.
- Nothing here can break anything. If things get strange, choose **Run → Clear state**, then run from the top.

> ⚠️ **Mock data only.** Everything in this course is fictional. Never load real offender data into any tool that isn't approved for criminal justice information.

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and builds today's data.
SESSION = 6

import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder in Databricks.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import env
PATHS = env.prepare(SESSION)

In [ ]:
import pandas as pd
from lnl import pipeline, text
from lnl.check import ___
pd.set_option("display.max_colwidth", 120)

df = pipeline.clean_through(4)
notes = text.sample_notes(df)          # 10,000 notes; everyone gets the same sample
notes[["note"]].head(15)

Officers write the same idea a hundred ways: *good*, *GOOD*, *goood*, *squared away*, *no issues*, *a pleasure to supervise*. And *good* shows up in notes that aren't praise at all: *not good*, *good time credit applied*, *C/O Goodman*.

## How we keep score
Every note in this sample has a true label (positive, negative, neutral, or mixed) from an answer key. For each attempt we measure:
- **Precision:** of the notes we returned, how many were actually good? Low precision means the list is padded with the wrong people.
- **Recall:** of all the good notes, how many did we find? Low recall means people are missing from the list.
- **F1:** one number that balances both. 1.0 is perfect.

In [ ]:
board = text.Scoreboard(notes["is_good"])
print(f"{notes['is_good'].sum():,} of {len(notes):,} notes are actually positive")

## Attempt 1: the word "good"

In [ ]:
attempt_1 = notes["note"].str.contains("good")
board.add("1. contains 'good'", attempt_1)

In [ ]:
text.mistakes(notes, attempt_1, "missed")

Each note is labeled by its TDCJ number, so anything a search gets wrong can be traced to an exact record. This one misses `Good` and `GOOD` (capital letters), and every note that says *excellent* or *no issues* instead.

## Attempt 2: ignore capitalization

In [ ]:
attempt_2 = notes["note"].str.contains("good", case=False)
board.add("2. any capitalization", attempt_2)

In [ ]:
text.mistakes(notes, attempt_2, "wrong", 10)

Recall went up, and so did the junk: *not good*, *good time credit* (a sentence credit, not praise), *Goodman* and *Goode* (officers), *goods* (property).

## Attempt 3: whole words only

In [ ]:
attempt_3 = notes["note"].str.contains(r"\bgood\b", case=False, regex=True)
board.add("3. whole word", attempt_3)

`\b` means "word boundary," so *Goodman* and *goods* no longer match. But neither does the typo *goood*, so recall dropped. **Every fix has a cost**, which is why we score each one instead of trusting our gut.

## Attempt 4: lemmas
A **lemma** is a word's dictionary form: *was* → *be*, *confiscated* → *confiscate*. The spaCy library works these out using grammar.

In [ ]:
nlp = text.load_spacy()
tokens = text.tokenize(notes["note"], nlp)       # about 10 seconds

example = "Behavior was better than last week. Contraband goods confiscated."
pd.DataFrame(text.tokenize(pd.Series([example]), nlp)[0], columns=["word", "lemma"]).T

Two surprises: *better* → *well* (not *good*), and *goods* → *good*. Lemmatizing just turned contraband into praise.

In [ ]:
attempt_4 = [any(lemma == "good" for _, lemma in note) for note in tokens]
board.add("4. lemma = good", attempt_4)

Lemmas normalize **grammar**, not **meaning**. On their own they don't solve this.

## Attempt 5: a vocabulary of "good"
🧪 **Your turn #1.** Build a list of words (matched by lemma) and phrases that mean good behavior. Run the cell, read the misses, add more, and run it again. Try to get recall above 0.9.

In [ ]:
MY_WORDS = {"good", "great", "excellent", ___}      # single words
MY_PHRASES = ["no issues", ___]                      # multi-word phrases

MY_WORDS = {w for w in MY_WORDS if isinstance(w, str)}
MY_PHRASES = [p for p in MY_PHRASES if isinstance(p, str)]
attempt_5 = text.lexicon_search(notes["note"], tokens, words=MY_WORDS, phrases=MY_PHRASES)
display(board.add("5. my vocabulary", attempt_5))
text.mistakes(notes, attempt_5, "missed", 10)

Compare yours with the full list the course uses:

In [ ]:
print(sorted(text.GOOD_WORDS))
print(text.GOOD_PHRASES)
attempt_5b = text.lexicon_search(notes["note"], tokens)
board.add("5b. full vocabulary", attempt_5b)

Recall jumps. Precision still suffers, because *not cooperative* contains *cooperative*.

## Attempt 6: handle negation
Rule: skip a match if *not*, *no*, *never*, *n't*, *anything but*, *far from*, or *no longer* appears within a few words before it.

In [ ]:
attempt_6 = text.lexicon_search(notes["note"], tokens, negation_window=3)
board.add("6. + negation", attempt_6)

🧪 **Your turn #2.** Change how far back the negation check looks. Try 1, then 6. Which scores better, and why? (No checker here; talk it over.)

In [ ]:
NEGATION_WINDOW = 3      # try 1, then 6
trial = text.lexicon_search(notes["note"], tokens, negation_window=NEGATION_WINDOW)
text.score(trial, notes["is_good"])

## Attempt 7: domain look-alikes
Some matches need knowledge of the business: *good time* and *good conduct time* are TDCJ sentence credits, Goodman and Goode are officers, and *goods* are property.

In [ ]:
attempt_7 = text.best_rules(notes["note"], tokens)
display(board.add("7. + domain exclusions", attempt_7))
text.mistakes(notes, attempt_7, "wrong", 10)

What's still "wrong" is mostly **mixed** notes: *good in the morning but combative by evening*. Is that a good-conduct note? That's a **business decision**, and it should be made before anyone builds the search, not after.

What's still *missed* is typos (*goood*, *gud*, *sqquared*) and phrasings nobody put on the list (*used powers only when authorized*).

## The scoreboard

In [ ]:
display(board.show())
board.plot();

## Wrap-up
- We went from an F1 around 0.1 to around 0.9. It took seven rounds, a hand-built vocabulary, negation rules, and domain knowledge.
- Rules are transparent and easy to audit, which matters for anything that affects a person.
- But they're brittle: the next new phrasing (*"kept his nose clean"*) slips through until someone adds it.

**Next session:** searching by **meaning** instead of words, and learning from a few hundred labeled examples instead of writing rules by hand.

---
## Answer key
Any vocabulary that gets recall above about 0.9 is a good answer. The course's full list is in `text.GOOD_WORDS` and `text.GOOD_PHRASES`. On the negation window: 1 misses *anything but good* and *not a good day*; 6 starts catching negators that belong to a different part of the sentence.